In [1]:
import pandas as pd

# Load Week 3 output (enriched dataset)
import pandas as pd

sold = pd.read_csv("../output/sold_combined_residential.csv", low_memory=False)
listings = pd.read_csv("../output/listings_combined_residential.csv", low_memory=False)

print("Sold shape:", sold.shape)
print("Listings shape:", listings.shape)

sold.head()

Sold shape: (397603, 84)
Listings shape: (540183, 84)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,94401,6472.0,NaN,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
1,SanDiego,SanDiego,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,...,91950,NaN,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,...,91950,NaN,NaN,NaN,CRMLS,CRMLS_SAND,NaN,NaN,NaN,NaN
3,Mlslistings,Mlslistings,NaN,False,NaN,NaN,NaN,NaN,1079166779,davidmartz@compass.com,...,92262,NaN,13504.0,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN
4,Southland,Southland,NaN,False,NaN,NaN,False,1890500.0,1075037759,karen.klein@theagencyre.com,...,91356,0.0,17873.0,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN


In [2]:
import os
print(os.listdir("../week1"))
print(os.listdir(".."))

['sold_aggregation.py', 'listings_aggregation.py']
['sold_aggregation.py', '.DS_Store', 'crmls_sold.py', 'week6', 'week1', 'week0', 'week7', 'output', 'combined_movies_data_standardized.csv', 'README.md', '.gitignore', 'week2', 'week5', '.venv', 'listings_aggregation.py', 'week4', '.git', 'eda_test.ipynb', 'venv311', 'crmls_listed.py', 'raw', 'eda_analysis.ipynb']


In [3]:
# Step 1: Convert date columns to datetime

date_cols_sold = ['CloseDate', 'ListingContractDate', 'PurchaseContractDate']
date_cols_listings = ['ListingContractDate']

for col in date_cols_sold:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors='coerce')

for col in date_cols_listings:
    if col in listings.columns:
        listings[col] = pd.to_datetime(listings[col], errors='coerce')

print("Date conversion complete")

# check result
sold[date_cols_sold].head()

Date conversion complete


,CloseDate,ListingContractDate,PurchaseContractDate
0,2024-01-26,2021-10-06,2023-11-22
1,2024-01-05,2021-03-08,2021-06-30
2,2024-01-05,2021-03-08,2021-11-18
3,2024-01-30,2024-01-30,2024-08-05
4,2024-01-29,2024-01-29,2024-01-29


In [4]:
# Step 2: Convert numeric columns

numeric_cols = [
    'ClosePrice', 'ListPrice', 'OriginalListPrice',
    'LivingArea', 'LotSizeAcres',
    'BedroomsTotal', 'BathroomsTotalInteger',
    'DaysOnMarket'
]

for col in numeric_cols:
    if col in sold.columns:
        sold[col] = pd.to_numeric(sold[col], errors='coerce')

print("Numeric conversion complete")

# check
sold[numeric_cols].head()

Numeric conversion complete


,ClosePrice,ListPrice,OriginalListPrice,LivingArea,LotSizeAcres,BedroomsTotal,BathroomsTotalInteger,DaysOnMarket
0,240000.0,295000.0,499000.0,1140.0,NaN,2.0,2.0,777
1,815000.0,759900.0,759900.0,1974.0,NaN,4.0,4.0,33
2,810000.0,770000.0,739900.0,1974.0,NaN,4.0,4.0,228
3,858000.0,858000.0,NaN,1995.0,0.3100,0.0,3.0,0
4,1890500.0,1890500.0,1890500.0,3194.0,0.4103,5.0,3.0,0


In [5]:
# Step 3: Remove invalid values

before_rows = len(sold)

sold = sold[
    (sold['ClosePrice'] > 0) &
    (sold['LivingArea'] > 0) &
    (sold['DaysOnMarket'] >= 0) &
    (sold['BedroomsTotal'] >= 0) &
    (sold['BathroomsTotalInteger'] >= 0)
]

after_rows = len(sold)

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before: 397603
Rows after: 397123
Rows removed: 480


In [6]:
# Step 4: Date COnsistency Flags

sold['listing_after_close_flag'] = sold['ListingContractDate'] > sold['CloseDate']
sold['purchase_after_close_flag'] = sold['PurchaseContractDate'] > sold['CloseDate']
sold['negative_timeline_flag'] = sold['PurchaseContractDate'] < sold['ListingContractDate']

print(sold[['listing_after_close_flag',
            'purchase_after_close_flag',
            'negative_timeline_flag']].sum())

listing_after_close_flag      58
purchase_after_close_flag    240
negative_timeline_flag       260
dtype: int64


In [7]:
# Step 5: Geographic Flags

sold['missing_coord_flag'] = sold['Latitude'].isna() | sold['Longitude'].isna()
sold['zero_coord_flag'] = (sold['Latitude'] == 0) | (sold['Longitude'] == 0)
sold['invalid_longitude_flag'] = sold['Longitude'] > 0

print(sold[['missing_coord_flag',
            'zero_coord_flag',
            'invalid_longitude_flag']].sum())

missing_coord_flag        15821
zero_coord_flag              25
invalid_longitude_flag       29
dtype: int64


In [9]:
# Step 6:  Saved final Clean Dataset

sold.to_csv("../output/sold_cleaned.csv", index=False)
print("Week 4 DONE")

Week 4 DONE
